# 🕸️ Web Scraping with Python — Notebook 1
### `requests` + `BeautifulSoup4` | Beginner to Intermediate

---

## 📌 What is the Web? (Start from Zero)

Every website you visit is made up of **HTML files** stored on a computer called a **server**.
When you type a URL in your browser:

```
Step 1 → Your browser sends a REQUEST to the server
Step 2 → The server sends back an HTML file (RESPONSE)
Step 3 → Your browser reads the HTML and displays it beautifully
```

**Web Scraping** = doing the same thing with Python code, but instead of displaying the HTML, we **extract useful data** from it.

---

## 🧱 What is HTML? (The Language of Webpages)

HTML uses **tags** to structure content. Think of tags like labels:

```html
<h1>This is a Heading</h1>          ← Big heading
<p>This is a paragraph.</p>         ← Normal text
<a href="/about">Click me</a>       ← Clickable link
<img src="photo.jpg">               ← An image
<div class="card">...</div>         ← A container/box
```

Every tag has:
- A **name** → `h1`, `p`, `a`, `div`
- Optional **attributes** → `href`, `class`, `id`, `src`
- **Content** → the text or other tags inside it

---

## ⚠️ Before You Scrape — Golden Rules

| Rule | Why? |
|------|------|
| Check `site.com/robots.txt` | It tells which pages you can/cannot scrape |
| Don't send 100s of requests per second | You'll overload (or get banned from) the server |
| Don't scrape personal data | Legal and ethical issue |

**Safe practice websites** (made for learners):
- 🔗 `https://quotes.toscrape.com`
- 🔗 `https://books.toscrape.com`
- 🔗 `https://httpbin.org` (for testing)

In [1]:
# ============================================================
# CELL 2 — Setup: Install & Import Libraries
# Run this once before starting
# ============================================================

# Uncomment and run this line if libraries are not installed yet
# !pip install requests beautifulsoup4

import requests                    # Sends HTTP requests (fetches webpages)
from bs4 import BeautifulSoup      # Reads and parses HTML

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 🔎 How to Find the Right Tag on Any Website — Inspect Element

Knowing BS4 syntax is only half the job. The other half is knowing **which tag and class to target** on a real website. That's where **Inspect Element** comes in — it's your most important tool as a web scraper.

### Step-by-Step: How to Inspect Any Element

1. Open any website in Chrome or Firefox
2. Right-click on the element you want to scrape (a price, a title, etc.)
3. Click **"Inspect"** or **"Inspect Element"**
4. The DevTools panel opens — the highlighted HTML is the element you clicked
5. Look for the **tag name**, **class**, and **id** of that element

```
You see on screen:          In HTML (DevTools shows):

  Python Crash Course    ->  <h2 class="book-title">Python Crash Course</h2>
  Rs 499                 ->  <p class="price">Rs 499</p>
  [View Details]         ->  <a href="/book/123" class="btn">View Details</a>
```

6. Now write your BS4 code:
```python
title = soup.find('h2', class_='book-title').text
price = soup.find('p',  class_='price').text
link  = soup.find('a',  class_='btn').get('href')
```

### Shortcut: Copy CSS Selector
In DevTools, right-click any HTML element → **Copy → Copy selector**  
Paste it directly into `soup.select_one(...)` — instant working code!

> **Practice this right now**: Open `https://quotes.toscrape.com`, right-click a quote text, and inspect it. Find its tag and class. Then try to scrape it using what you've learned.

## 🌐 Fetching a Webpage — `requests.get()`

`requests.get(url)` is all you need to fetch a webpage. It returns a **response object** that contains everything the server sent back.

For now, focus on just **two things** from the response:

```python
response.status_code   # Did the request succeed?
response.text          # What is the actual content (HTML)?
```

### What is a Status Code?
It's the server's way of telling you if your request worked:

```
200  →  ✅ OK — page fetched successfully
404  →  ❌ Not Found — wrong URL
403  →  ❌ Forbidden — server blocked you
500  →  ❌ Server Error — problem on website's side
```

> 💡 **Always check `status_code == 200` before processing the response.**

In [2]:
# ============================================================
# CELL 3 — Your First Web Request
# ============================================================

url = "https://quotes.toscrape.com"

response = requests.get(url)          # Send request, get response

print("Status Code:", response.status_code)   # 200 means success ✅

# response.text contains the entire HTML of the page
# It's a very long string — let's just print the first 500 characters
print("\nFirst 500 characters of the HTML:")
print("-" * 50)
print(response.text[:500])

Status Code: 200

First 500 characters of the HTML:
--------------------------------------------------
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div cla


## 🔍 Looking at Raw HTML — What Did We Get?

The `response.text` you printed above looks messy and hard to read. That's **raw HTML** — the same code your browser receives, but your browser hides it and shows you the pretty version.

### View Page Source — Try This!

Go to `https://quotes.toscrape.com` in your browser and press:
- **Windows/Linux**: `Ctrl + U`
- **Mac**: `Cmd + Option + U`

You'll see the exact same HTML that `response.text` contains. Our job is to **extract the useful parts** from this HTML.

---

### Checking the Response Before Using It

Always verify the request succeeded before trying to extract data:

```python
if response.status_code == 200:
    # safe to process the HTML
else:
    print("Something went wrong!")
```

In [3]:
# ============================================================
# CELL 4 — Checking Response Before Processing
# ============================================================

url = "https://quotes.toscrape.com"
response = requests.get(url)

# --- Always check status code first ---
if response.status_code == 200:
    print("✅ Page fetched successfully!")
    print(f"Total HTML length: {len(response.text)} characters")
else:
    print(f"❌ Failed! Status code: {response.status_code}")

# --- Try a page that doesn't exist ---
bad_url = "https://quotes.toscrape.com/this-page-does-not-exist"
bad_response = requests.get(bad_url)

print(f"\nBad URL status code: {bad_response.status_code}")   # Will be 404

if bad_response.status_code == 200:
    print("Got the page")
else:
    print(f"❌ Page not found (status: {bad_response.status_code})")

✅ Page fetched successfully!
Total HTML length: 11021 characters

Bad URL status code: 404
❌ Page not found (status: 404)


## 🍲 Introducing BeautifulSoup — Making Sense of HTML

Raw HTML is just a long string of text — hard to work with directly.
**BeautifulSoup** converts that string into a structured tree that you can easily navigate and search.

### The 2-Step Pattern (You'll Use This Every Time)

```python
# Step 1: Fetch the HTML
response = requests.get(url)

# Step 2: Parse the HTML with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')
```

That's it. Now `soup` is a smart object that understands HTML structure.

### What is `html.parser`?

It's the **engine** that reads the HTML string and builds the tree. Python has it built-in — no extra installation needed. Always use `'html.parser'` as a beginner.

```
Raw HTML string  ──►  BeautifulSoup('html.parser')  ──►  Navigable Tree
"<h1>Hello</h1>"                                        soup.h1.text = "Hello"
```

In [4]:
# ============================================================
# CELL 5 — Creating a BeautifulSoup Object
# ============================================================

# We'll use a small, simple HTML string to learn
# (easier to understand than a full webpage at first)

simple_html = """
<html>
  <head>
    <title>My First Page</title>
  </head>
  <body>
    <h1>Hello, Web Scraping!</h1>
    <p>This is a paragraph.</p>
    <p>This is another paragraph.</p>
  </body>
</html>
"""

# Create the soup object
soup = BeautifulSoup(simple_html, 'html.parser')

# prettify() → prints the HTML in a clean, indented format
# Great for visualizing the structure
print(soup.prettify())

<html>
 <head>
  <title>
   My First Page
  </title>
 </head>
 <body>
  <h1>
   Hello, Web Scraping!
  </h1>
  <p>
   This is a paragraph.
  </p>
  <p>
   This is another paragraph.
  </p>
 </body>
</html>



## 🧭 Accessing Tags Directly

Once you have a `soup` object, you can access any HTML tag just like a Python attribute:

```python
soup.title    # The <title> tag
soup.h1       # The first <h1> tag
soup.p        # The first <p> tag
```

> ⚠️ **Important**: `soup.tag_name` always returns the **first** matching tag only.

### Getting the Text Out of a Tag

```python
soup.h1            # Returns the full tag:  <h1>Hello, Web Scraping!</h1>
soup.h1.text       # Returns just the text: Hello, Web Scraping!
```

| What you write | What you get |
|---|---|
| `soup.title` | `<title>My First Page</title>` |
| `soup.title.text` | `'My First Page'` |
| `soup.h1` | `<h1>Hello, Web Scraping!</h1>` |
| `soup.h1.text` | `'Hello, Web Scraping!'` |
| `soup.p` | `<p>This is a paragraph.</p>` (first `<p>` only) |

In [6]:
# ============================================================
# CELL 6 — Accessing Tags and Getting Text
# ============================================================

# Using the same soup from Cell 5

# --- Access full tag (includes HTML) ---
print("Full tag:")
print(soup.title)              # <title>My First Page</title>
print(soup.h1)      
print(soup.p)           # <h1>Hello, Web Scraping!</h1>
print()

# --- Extract just the text (.text) ---
print("Just the text:")
print(soup.title.text)         # My First Page
print(soup.h1.text)            # Hello, Web Scraping!
print()

# --- soup.p gives only the FIRST <p> ---
print("First <p> tag:")
print(soup.p.text)             # This is a paragraph.
                               # (second <p> is ignored)
print()

# --- Checking if a tag exists ---
print("Does <h2> exist?", soup.h2)    # None — because there's no <h2> in our HTML

Full tag:
<title>My First Page</title>
<h1>Hello, Web Scraping!</h1>
<p>This is a paragraph.</p>

Just the text:
My First Page
Hello, Web Scraping!

First <p> tag:
This is a paragraph.

Does <h2> exist? None


## 🏗️ Understanding HTML Attributes — `class`, `id`, `href`

HTML tags can have **attributes** — extra information attached to a tag:

```html
<p class="price">₹499</p>              ← class attribute
<div id="main-content">...</div>        ← id attribute
<a href="https://google.com">Link</a>   ← href attribute
<img src="photo.jpg" alt="A photo">     ← src and alt attributes
```

### Why Do Attributes Matter for Scraping?

On a real webpage, there are **many `<p>` tags** — some are prices, some are descriptions, some are footers. Attributes (`class`, `id`) help you **target exactly the tag you want**.

### Reading Attributes in BS4

```python
tag.attrs          # Returns ALL attributes as a dictionary
tag['href']        # Read a specific attribute (like a dict key)
tag.get('href')    # Safer — returns None instead of error if attribute missing
```

In [ ]:
# ============================================================
# CELL 7 — Reading Tag Attributes
# ============================================================

html_with_attrs = """
<html><body>
  <h1 id="main-title">Book Store</h1>
  <p class="price">₹499</p>
  <p class="description">A great Python book.</p>
  <a href="https://python.org" target="_blank">Visit Python</a>
  <img src="cover.jpg" alt="Book Cover">
</body></html>
"""

soup2 = BeautifulSoup(html_with_attrs, 'html.parser')

# --- .attrs → gives all attributes as a dict ---
print("All attributes of <h1>:", soup2.h1.attrs)    # {'id': 'main-title'}
print("All attributes of <a> :", soup2.a.attrs)     # {'href': ..., 'target': ...}
print()

# --- Access a specific attribute ---
print("href of <a>  :", soup2.a['href'])            # https://python.org
print("src of <img> :", soup2.img['src'])           # cover.jpg
print("alt of <img> :", soup2.img['alt'])           # Book Cover
print()

# --- .get() is safer (no crash if attribute is missing) ---
print("id of <h1>   :", soup2.h1.get('id'))        # main-title
print("id of <p>    :", soup2.p.get('id'))         # None (no id on <p>)


All attributes of <h1>: {'id': 'main-title'}
All attributes of <a> : {'href': 'https://python.org', 'target': '_blank'}

href of <a>  : https://python.org
src of <img> : cover.jpg
alt of <img> : Book Cover

id of <h1>   : main-title
id of <p>    : None
main-title


## 🎯 `find()` — Search for a Specific Tag

`soup.p` gives you the first `<p>` — but what if you want a `<p>` with a specific class?
That's where `find()` comes in.

### Syntax

```python
soup.find('tag_name')                    # Find first tag by name
soup.find('tag_name', class_='...')      # Find first tag with a specific class
soup.find('tag_name', id='...')          # Find first tag with a specific id
```

> ⚠️ Note: we write `class_` (with underscore) because `class` is a reserved keyword in Python.

### `find()` returns:
- The **first matching tag** if found
- **`None`** if no match is found (no error!)

```python
# On a webpage with multiple <p> tags:
soup.find('p')                    # First <p> — could be anything
soup.find('p', class_='price')    # First <p> that has class="price" ✅
```

In [10]:
# ============================================================
# CELL 8 — find(): Search by Tag, Class and ID
# ============================================================

bookstore_html = """
<html><body>
  <div id="book1">
    <h2>Python Crash Course</h2>
    <p class="price">₹499</p>
    <p class="author">Eric Matthes</p>
  </div>
  <div id="book2">
    <h2>Automate the Boring Stuff</h2>
    <p class="price">₹399</p>
    <p class="author">Al Sweigart</p>
  </div>
</body></html>
"""

soup3 = BeautifulSoup(bookstore_html, 'html.parser')

# --- Find by tag name (gives first match) ---
print(soup3.find('h2').text)                       # Python Crash Course

# --- Find by class ---
print(soup3.find('p', class_='price').text)        # ₹499  (first price found)
print(soup3.find('p', class_='author').text)       # Eric Matthes

# --- Find by id ---
print(soup3.find('div', id='book2').find('h2').text)   # Automate the Boring Stuff
                                                        # ↑ find inside another find!

# --- Returns None if not found (no crash) ---
result = soup3.find('p', class_='rating')
print(result)    # None — class 'rating' doesn't exist in our HTML

Python Crash Course
₹499
Eric Matthes
Automate the Boring Stuff
None


## 📋 `find_all()` — Find Every Matching Tag

`find()` gives you **one** result. `find_all()` gives you **all** results as a **list**.

```python
soup.find_all('tag')               # All tags with that name → list
soup.find_all('tag', class_='...')  # All tags with that class → list
```

### The Loop Pattern

Since `find_all()` returns a list, you loop through it:

```python
all_prices = soup.find_all('p', class_='price')

for price in all_prices:
    print(price.text)
```

### `find()` vs `find_all()` — Quick Comparison

| | `find()` | `find_all()` |
|--|----------|-------------|
| Returns | First match (one tag) | All matches (list) |
| If not found | `None` | Empty list `[]` |
| Use when | You need 1 result | You need multiple results |

In [11]:
# ============================================================
# CELL 9 — find_all(): Get All Matching Tags
# ============================================================

# Using soup3 from Cell 8 (bookstore HTML)

# --- find_all() by tag name ---
all_titles = soup3.find_all('h2')
print(f"Found {len(all_titles)} book titles:")
for title in all_titles:
    print(" •", title.text)
print()

# --- find_all() by class ---
all_prices = soup3.find_all('p', class_='price')
print(f"Found {len(all_prices)} prices:")
for price in all_prices:
    print(" •", price.text)
print()

# --- Collecting data from all books ---
print("All books structured:")
print("-" * 35)
all_books = soup3.find_all('div')    # Get all <div> blocks
for book in all_books:
    title  = book.find('h2').text
    price  = book.find('p', class_='price').text
    author = book.find('p', class_='author').text
    print(f"📖 {title}")
    print(f"   Price  : {price}")
    print(f"   Author : {author}")
    print()

Found 2 book titles:
 • Python Crash Course
 • Automate the Boring Stuff

Found 2 prices:
 • ₹499
 • ₹399

All books structured:
-----------------------------------
📖 Python Crash Course
   Price  : ₹499
   Author : Eric Matthes

📖 Automate the Boring Stuff
   Price  : ₹399
   Author : Al Sweigart



## ⚠️ None Safety + Common Beginner Mistakes

These are the mistakes **every beginner makes**. Learn them now and save hours of frustration.

### Mistake 1: Calling `.text` on `None` (Most Common Crash!)

`find()` returns `None` if the tag is not found. Calling `.text` on `None` crashes your code:

```python
# WRONG - crashes with AttributeError if tag not found
price = soup.find('p', class_='price').text

# RIGHT - always check first
tag = soup.find('p', class_='price')
price = tag.text if tag else 'N/A'
```

### Mistake 2: `class` instead of `class_`
```python
soup.find('p', class='price')    # SyntaxError! 'class' is a Python keyword
soup.find('p', class_='price')   # Correct - use class_ with underscore
```

### Mistake 3: Forgetting `.text` — Getting the Whole Tag Instead
```python
print(soup.find('h2'))        # <h2>Python Crash Course</h2>  <- full tag!
print(soup.find('h2').text)   # Python Crash Course           <- just text
```

### Mistake 4: `find_all()` Returns `[]` — Crashing on Index
```python
results = soup.find_all('p', class_='rating')  # returns []
print(results[0].text)   # IndexError! list is empty

# Right way:
if results:
    print(results[0].text)
```

In [ ]:
# CELL - None Safety and Common Mistakes in Action

demo_html = '''
<div>
  <h2>Python Book</h2>
  <p class="price">Rs 499</p>
</div>
'''
soup_demo = BeautifulSoup(demo_html, 'html.parser')

# --- Mistake 1 Demo: find() returning None ---
tag = soup_demo.find('p', class_='rating')   # Does not exist!
print('find() returned:', tag)               # None

# WRONG (uncomment to see the crash):
# print(tag.text)   # AttributeError: 'NoneType' object has no attribute 'text'

# RIGHT - safe pattern
rating = tag.text if tag else 'Not Available'
print('Rating:', rating)   # Not Available
print()

# --- Safe extraction function (use this pattern always) ---
def safe_text(soup_obj, tag, **kwargs):
    """
    Safely extracts text from a tag.
    Returns default value instead of crashing if tag not found.
    """
    found = soup_obj.find(tag, **kwargs)
    return found.get_text(strip=True) if found else 'N/A'

print('Price  :', safe_text(soup_demo, 'p', class_='price'))    # Rs 499
print('Rating :', safe_text(soup_demo, 'p', class_='rating'))   # N/A
print('Author :', safe_text(soup_demo, 'p', class_='author'))   # N/A

# --- Mistake 4: Empty find_all() ---
print()
results = soup_demo.find_all('span')    # No <span> tags exist
print('find_all result:', results)       # []

if results:                             # Always guard before using
    print(results[0].text)
else:
    print('No results found - handled safely!')

## 🔗 Extracting Links — A Real Use Case

One of the most common scraping tasks is **extracting all links** from a page. Links are in `<a>` tags, and the actual URL is in the `href` attribute.

```html
<a href="https://example.com/page1">Page 1</a>
        ↑
   This is what we want
```

### The Pattern

```python
all_links = soup.find_all('a')         # Get all anchor tags

for link in all_links:
    url  = link.get('href')            # Extract href attribute
    text = link.text                   # Extract link text
```

> 💡 Use `.get('href')` instead of `['href']` — some `<a>` tags don't have an `href` and `['href']` would crash. `.get()` safely returns `None` instead.

In [ ]:
# ============================================================
# CELL 10 — Extracting Links from a Real Webpage
# ============================================================

# Fetch a real webpage
response = requests.get("https://quotes.toscrape.com")
soup_real = BeautifulSoup(response.text, 'html.parser')

# Find all <a> tags
all_links = soup_real.find_all('a')
print(f"Total <a> tags found: {len(all_links)}")
print()

# Extract href and link text from each
print("All links on the page:")
print("-" * 45)
for link in all_links:
    href = link.get('href')          # None if no href attribute
    text = link.text.strip()         # .strip() removes extra spaces/newlines

    if href:                         # Only print if href exists
        print(f"Text : {text}")
        print(f"URL  : {href}")
        print()

# 🎉 You just scraped your first real webpage!
print("✅ Batch 1 complete! You can now fetch pages and extract links.")

---
## 🎨 CSS Selectors — `select()` and `select_one()`

Besides `find()` and `find_all()`, BeautifulSoup also supports **CSS Selectors** — the same syntax used in CSS stylesheets.

| Method | Returns | Equivalent To |
|--------|---------|---------------|
| `soup.select_one(selector)` | First match | `find()` |
| `soup.select(selector)` | All matches (list) | `find_all()` |

### Selector Syntax — Learn These 5 Core Patterns

```
"h2"           ->  any <h2> tag
".price"       ->  any tag with  class="price"
"#book1"       ->  tag with      id="book1"
"p.price"      ->  <p> with      class="price"
"div.book h2"  ->  <h2> inside   <div class="book">  (anywhere inside)
```

> When to use `select()` vs `find_all()`? Both do the same job. Use `find_all()` for simple searches. Use `select()` when you need to target deeply nested elements more precisely.

In [12]:
# CELL 11 - CSS Selectors: select() and select_one()
# Using soup3 from Cell 8 (bookstore HTML)
# soup3 has <div id='book1'> and <div id='book2'>
# each with <h2>, <p class='price'>, <p class='author'>

# select_one() -> first match only
print(soup3.select_one('h2').text)               # Python Crash Course
print(soup3.select_one('.price').text)           # Rs 499
print(soup3.select_one('#book2 h2').text)        # Automate the Boring Stuff
print()

# select() -> all matches as a list
all_titles = soup3.select('h2')
for t in all_titles:
    print('Title:', t.text)
print()

# p.price means: <p> tag WITH class='price'
all_prices = soup3.select('p.price')
for p in all_prices:
    print('Price:', p.text)
print()

# Descendant selector — h2 anywhere inside a div
nested = soup3.select('div h2')
print(f'Found {len(nested)} titles using descendant selector')

Python Crash Course
₹499
Automate the Boring Stuff

Title: Python Crash Course
Title: Automate the Boring Stuff

Price: ₹499
Price: ₹399

Found 2 titles using descendant selector


## 🧹 Cleaning Extracted Text

When you scrape from real websites, text often comes with **extra spaces, newlines, or unwanted characters**. Cleaning it is essential.

### `.text` vs `.get_text()` — What is the Difference?

| | `.text` | `.get_text()` |
|--|---------|---------------|
| Returns | All text concatenated | Same, but customizable |
| Separator between nodes | None | You can set one |
| Strip whitespace | No | Yes, with `strip=True` |

### Common Cleaning Techniques

```python
tag.text.strip()                    # Remove edge whitespace
tag.get_text(strip=True)            # Same, via get_text
text.replace('Rs', '').strip()      # Remove specific characters
' '.join(text.split())              # Collapse all internal whitespace
```

In [13]:
# CELL 12 - Cleaning Scraped Text

messy_html = '''
<div class="product">
  <h2>  Python Crash Course  </h2>
  <p class="price">  Rs 499.00  </p>
  <p class="desc">
    A   great book
    for beginners.
  </p>
</div>
'''

soup_messy = BeautifulSoup(messy_html, 'html.parser')

# Problem: raw .text keeps extra whitespace
title_raw = soup_messy.find('h2').text
print(f"Raw title   : '{title_raw}'")

title_clean = soup_messy.find('h2').text.strip()
print(f"Clean title : '{title_clean}'")
print()

# Clean price: remove currency symbol
price_raw = soup_messy.find('p', class_='price').text
price_clean = price_raw.strip().replace('Rs', '').strip()
print(f"Raw price   : '{price_raw}'")
print(f"Clean price : '{price_clean}'")
print()

# get_text for multi-line text
desc = soup_messy.find('p', class_='desc')
print(f"Raw .text        : '{desc.text}'")
print(f"get_text strip   : '{desc.get_text(strip=True)}'")

# Best method: collapse all whitespace
desc_clean = ' '.join(desc.get_text().split())
print(f"Fully clean      : '{desc_clean}'")    # A great book for beginners.

Raw title   : '  Python Crash Course  '
Clean title : 'Python Crash Course'

Raw price   : '  Rs 499.00  '
Clean price : '499.00'

Raw .text        : '
    A   great book
    for beginners.
  '
get_text strip   : 'A   great book
    for beginners.'
Fully clean      : 'A great book for beginners.'


## 🌳 Navigating the HTML Tree

HTML is structured like a tree — tags are nested inside each other. BS4 lets you move in all directions.

```
         <div>              <- Parent
        /      \
      <h2>    <p>           <- Children (siblings of each other)
       |       |
    'Title'  '499'          <- Text content
```

| Property | What It Returns |
|----------|-----------------|
| `tag.parent` | The tag that wraps this tag |
| `list(tag.children)` | All direct children |
| `tag.find_next('p')` | Next `<p>` anywhere after this tag |
| `tag.next_sibling` | The immediately next sibling node |

> When is this useful? When you find a known tag (like a label) and want the **data sitting next to it**.

In [14]:
# CELL 13 - Tree Navigation: parent, children, find_next

tree_html = '''
<div class="card">
  <h2>Data Science with Python</h2>
  <p class="price">Rs 599</p>
  <p class="author">Wes McKinney</p>
</div>
'''

soup_tree = BeautifulSoup(tree_html, 'html.parser')
h2 = soup_tree.find('h2')

# .parent -> go UP the tree
print('Parent tag  :', h2.parent.name)             # div
print('Parent class:', h2.parent.get('class'))     # ['card']
print()

# .children -> direct children of a tag
div = soup_tree.find('div')
children = list(div.children)
# children includes whitespace text nodes - keep only real tags
tag_children = [c for c in children if c.name]
print('Children tags:', [c.name for c in tag_children])   # ['h2', 'p', 'p']
print()

# find_next() -> grab element right after a known tag
price_tag = soup_tree.find('p', class_='price')
next_p = price_tag.find_next('p')
print('Next <p> after price:', next_p.text)        # Wes McKinney

Parent tag  : div
Parent class: ['card']

Children tags: ['h2', 'p', 'p']

Next <p> after price: Wes McKinney


## 🔎 Advanced Searching — Multiple Filters, Lists and Regex

### Combine Multiple Attributes
```python
# <a> tags that have BOTH class='link' AND an href attribute present
soup.find_all('a', class_='link', href=True)
```

### Search Multiple Tag Names at Once
```python
soup.find_all(['h1', 'h2', 'h3'])    # All headings in one call
```

### `limit` — Stop After N Results
```python
soup.find_all('p', limit=3)          # Only the first 3 matches
```

### Regex in Searches
```python
import re
soup.find_all(re.compile('^h'))                  # All tags: h1, h2, h3 ...
soup.find_all('p', class_=re.compile('price'))   # class containing 'price'
```

In [15]:
# CELL 14 - Advanced Searching

adv_html = '''
<html><body>
  <h1>Books</h1>
  <h2>Fiction</h2>
  <h3>Bestsellers</h3>
  <p class="price-inr">Rs 499</p>
  <p class="price-usd">$6</p>
  <p class="author">Author Name</p>
  <a href="/book1">Book One</a>
  <a>No link here</a>
  <a href="/book2" class="featured">Book Two</a>
</body></html>
'''

soup_adv = BeautifulSoup(adv_html, 'html.parser')

# Multiple tag names at once
headings = soup_adv.find_all(['h1', 'h2', 'h3'])
print('All headings:', [h.text for h in headings])

# limit
first_two = soup_adv.find_all('p', limit=2)
print('First 2 <p>s:', [p.text for p in first_two])

# href=True: only <a> tags that HAVE an href attribute
links = soup_adv.find_all('a', href=True)
print('\n<a> with href:')
for a in links:
    print(f' - {a.text} -> {a["href"]}')

# Regex: all heading tags (h1, h2, h3 ...)
print('\nHeadings via regex:')
for tag in soup_adv.find_all(re.compile('^h[0-9]')):
    print(f' - <{tag.name}>: {tag.text}')

# Regex on class: any <p> with 'price' anywhere in class name
print('\nPrices via regex class:')
for p in soup_adv.find_all('p', class_=re.compile('price')):
    print(f' - {p.get("class")} -> {p.text}')

All headings: ['Books', 'Fiction', 'Bestsellers']
First 2 <p>s: ['Rs 499', '$6']

<a> with href:
 - Book One -> /book1
 - Book Two -> /book2

Headings via regex:


NameError: name 're' is not defined

## 🌐 Adding Headers — Why Your Scraper Gets Blocked

When `requests` sends a GET request, servers see who is asking. By default, Python sends:

```
User-Agent: python-requests/2.x.x
```

Many websites **block this**. The fix is to add a browser-like `User-Agent` header.

### Query Parameters — Navigating Pages

Instead of building URLs manually like `?page=2&sort=price`, use the `params` argument:

```python
# These two lines are identical:
requests.get('https://site.com/books?page=2&sort=price')
requests.get('https://site.com/books', params={'page': 2, 'sort': 'price'})
```

Using `params={}` is cleaner — requests handles URL-encoding automatically.

In [16]:
# CELL 15 - Headers and Query Parameters

# Without headers - Python's default User-Agent
r1 = requests.get('https://quotes.toscrape.com')
print('Default User-Agent:', r1.request.headers['User-Agent'])
print('Status:', r1.status_code)
print()

# With a browser-like User-Agent
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
    )
}
r2 = requests.get('https://quotes.toscrape.com', headers=headers)
print('Browser User-Agent:', r2.request.headers['User-Agent'][:55], '...')
print('Status:', r2.status_code)
print()

# Query params to fetch page 2
r3 = requests.get('https://quotes.toscrape.com/page/2/', headers=headers)
print('Page 2 URL   :', r3.url)
print('Page 2 Status:', r3.status_code)

Default User-Agent: python-requests/2.32.3
Status: 200

Browser User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/5 ...
Status: 200

Page 2 URL   : https://quotes.toscrape.com/page/2/
Page 2 Status: 200


## 🚨 Error Handling — Make Your Scraper Crash-Proof

Without error handling, one bad URL crashes the entire script and you lose all collected data.

### Two Types of Errors You Must Handle

**1. HTTP Errors** — Server replied but with a bad status code (403, 404, 500):
```python
response.raise_for_status()   # Raises HTTPError for 4xx and 5xx codes
```

**2. Connection Errors** — Server did not reply at all:
```python
requests.get(url, timeout=10)   # Wait max 10 seconds
```

### The Standard Safe-Request Pattern
```python
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    # safe to process here
except requests.exceptions.RequestException as e:
    print(f'Failed: {e}')
```

In [17]:
# CELL 16 - Error Handling with try/except

def fetch_page(url, headers=None):
    """
    Safely fetches a URL and returns a BeautifulSoup object.
    Returns None if anything goes wrong - no crash!
    """
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()   # Error on 4xx / 5xx
        return BeautifulSoup(response.text, 'html.parser')

    except requests.exceptions.HTTPError as e:
        print(f'  HTTP Error  : {e}')
    except requests.exceptions.ConnectionError:
        print(f'  Cannot reach: {url}')
    except requests.exceptions.Timeout:
        print(f'  Timed out   : {url}')
    except requests.exceptions.RequestException as e:
        print(f'  Failed      : {e}')

    return None


# Test 1: Working URL
print('Test 1: Valid URL')
soup_ok = fetch_page('https://quotes.toscrape.com')
if soup_ok:
    print('  OK! Page title:', soup_ok.title.text.strip())

print()

# Test 2: Non-existent page
print('Test 2: Bad URL (should fail gracefully)')
soup_bad = fetch_page('https://quotes.toscrape.com/this-does-not-exist')
if soup_bad is None:
    print('  Handled gracefully - no crash!')

Test 1: Valid URL
  OK! Page title: Quotes to Scrape

Test 2: Bad URL (should fail gracefully)
  HTTP Error  : 404 Client Error: NOT FOUND for url: https://quotes.toscrape.com/this-does-not-exist
  Handled gracefully - no crash!


## 📊 Scraping HTML Tables

Tables on websites hold structured data — stock prices, scores, product listings. They follow a predictable structure, so scraping them is very systematic.

### HTML Table Tags to Know

```html
<table>           <- the whole table
  <thead>         <- header section
    <tr>          <- header row
      <th>Name</th><th>Price</th>
    </tr>
  </thead>
  <tbody>         <- data section
    <tr>          <- one data row
      <td>Python Book</td><td>Rs 499</td>
    </tr>
  </tbody>
</table>
```

### Strategy
1. `find_all('th')` -> column names
2. `find('tbody').find_all('tr')` -> all data rows
3. `row.find_all('td')` -> cells in each row
4. `zip(headers, cells)` -> combine into a dict

In [18]:
# CELL 17 - Scraping HTML Tables

table_html = '''
<table>
  <thead>
    <tr><th>Book</th><th>Author</th><th>Price</th><th>Rating</th></tr>
  </thead>
  <tbody>
    <tr><td>Python Crash Course</td><td>Eric Matthes</td><td>Rs 499</td><td>4.8</td></tr>
    <tr><td>Automate Boring Stuff</td><td>Al Sweigart</td><td>Rs 399</td><td>4.7</td></tr>
    <tr><td>Fluent Python</td><td>Luciano Ramalho</td><td>Rs 799</td><td>4.9</td></tr>
  </tbody>
</table>
'''

soup_table = BeautifulSoup(table_html, 'html.parser')

# Step 1: column headers
col_headers = [th.text for th in soup_table.find_all('th')]
print('Columns:', col_headers)
print()

# Step 2: all data rows
rows = soup_table.find('tbody').find_all('tr')

# Step 3 & 4: build list of dicts
books = []
for row in rows:
    cells = [td.text.strip() for td in row.find_all('td')]
    book_dict = dict(zip(col_headers, cells))
    books.append(book_dict)

print(f'Scraped {len(books)} books:')
for book in books:
    print(book)

Columns: ['Book', 'Author', 'Price', 'Rating']

Scraped 3 books:
{'Book': 'Python Crash Course', 'Author': 'Eric Matthes', 'Price': 'Rs 499', 'Rating': '4.8'}
{'Book': 'Automate Boring Stuff', 'Author': 'Al Sweigart', 'Price': 'Rs 399', 'Rating': '4.7'}
{'Book': 'Fluent Python', 'Author': 'Luciano Ramalho', 'Price': 'Rs 799', 'Rating': '4.9'}


## 💾 Saving Scraped Data — CSV and JSON

| Format | Best For | Open With |
|--------|----------|-----------|
| CSV | Tabular data (rows and columns) | Excel, Pandas, Google Sheets |
| JSON | Nested or flexible data | Any language, REST APIs |

### CSV
```python
import csv
with open('data.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['Book', 'Price'])
    writer.writeheader()
    writer.writerows(data)
```

### JSON
```python
import json
with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)
# ensure_ascii=False -> keeps Rs symbol, Hindi text etc.
```

In [ ]:
# CELL 18 - Saving Data to CSV and JSON

import csv
import json

# Using books list from Cell 17

# --- Save to CSV ---
csv_file = 'books.csv'
with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=books[0].keys())
    writer.writeheader()
    writer.writerows(books)
print(f'Saved to {csv_file}')

# --- Save to JSON ---
json_file = 'books.json'
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(books, f, indent=4, ensure_ascii=False)
print(f'Saved to {json_file}')
print()

# Verify: read back the JSON
with open(json_file, 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print('JSON preview:')
print(json.dumps(loaded, indent=2, ensure_ascii=False))

## ⏱️ Two Must-Know Practical Skills

### Skill 1: Rate Limiting with `time.sleep()`

When scraping multiple pages, sending requests too fast will get you **blocked by the website**. Always add a delay between requests:

```python
import time

for page in range(1, 6):                      # Scraping 5 pages
    response = requests.get(f'/page/{page}/')
    # ... scrape the page ...
    time.sleep(1)                              # Wait 1 second before next request
```

Use `random.uniform(1, 3)` for random delays — more human-like:
```python
import random
time.sleep(random.uniform(1, 3))    # Wait between 1 and 3 seconds randomly
```

### Skill 2: Converting Scraped Strings to Numbers

All scraped data comes as **strings**. If you want to do math (sort by price, calculate average), you must convert:

```python
# Scraped text:  'Rs 1,299.00'
# Goal:          1299.0  (a float)

price_str = 'Rs 1,299.00'
price_num = float(
    price_str
    .replace('Rs', '')   # Remove currency symbol
    .replace(',', '')    # Remove thousands separator
    .strip()             # Remove whitespace
)                        # -> 1299.0
```

In [ ]:
# CELL - Rate Limiting and String-to-Number Conversion

import time
import random

# --- Rate limiting demo ---
urls = [
    'https://quotes.toscrape.com/page/1/',
    'https://quotes.toscrape.com/page/2/',
    'https://quotes.toscrape.com/page/3/',
]

print('Scraping with polite delays...')
for url in urls:
    r = requests.get(url)
    print(f'  Fetched: {url} | Status: {r.status_code}')
    delay = random.uniform(1, 2)         # Random delay between 1-2 seconds
    print(f'  Sleeping {delay:.1f}s...')
    time.sleep(delay)

print('Done!')
print()

# --- String to Number conversion ---
scraped_prices = ['Rs 499.00', 'Rs 1,299.00', 'Rs 99.99']

print('String -> Number conversion:')
numeric_prices = []
for p in scraped_prices:
    num = float(p.replace('Rs', '').replace(',', '').strip())
    numeric_prices.append(num)
    print(f"  '{p}' -> {num}")

print()
print(f'Cheapest : Rs {min(numeric_prices)}')
print(f'Most Exp : Rs {max(numeric_prices)}')
print(f'Average  : Rs {sum(numeric_prices)/len(numeric_prices):.2f}')

## Quick Reference — Cheat Sheet

Everything learned so far in one place.

### Core Pattern
```python
import requests
from bs4 import BeautifulSoup

headers = {'User-Agent': 'Mozilla/5.0 ...'}
response = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(response.text, 'html.parser')
```

### Searching
```python
soup.find('tag')                             # First match
soup.find('tag', class_='name')              # By class
soup.find('tag', id='name')                  # By id
soup.find_all('tag')                         # All matches -> list
soup.find_all(['h1','h2','h3'])              # Multiple tag names
soup.find_all('tag', limit=5)                # First 5 only
soup.find_all('a', href=True)                # Tags with href present
soup.find_all('p', class_=re.compile('x'))  # Regex on class
```

### CSS Selectors
```python
soup.select_one('p.price')                  # First <p class='price'>
soup.select('div.card h2')                  # All <h2> inside div.card
soup.select('a[href]')                      # <a> tags that have href
```

### Extracting Data
```python
tag.text                                    # All text
tag.get_text(strip=True)                    # Text without whitespace
' '.join(tag.text.split())                 # Collapse internal spaces
tag.get('href')                             # Attribute safely
tag.attrs                                   # All attributes as dict
```

### Tree Navigation
```python
tag.parent                                  # One level up
list(tag.children)                          # Direct children
tag.find_next('p')                          # Next <p> after this tag
```

### Error Handling
```python
try:
    r = requests.get(url, timeout=10)
    r.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f'Failed: {e}')
```

### Encoding Fix (for non-English sites)
```python
response.encoding = 'utf-8'   # Force correct encoding before reading .text
```

### URL Joining
```python
from urllib.parse import urljoin
urljoin(base_url, relative_path)  # Safe URL building
```

### Saving with Pandas (optional)
```python
import pandas as pd
pd.DataFrame(data).to_csv('output.csv', index=False)  # Easier than csv module
```

### Session
```python
session = requests.Session()
session.headers.update(headers)  # Set once for all requests
session.get(url)                 # Headers applied automatically
session.close()
```

---
# 🧪 Mini Projects — Putting It All Together

You now know all the building blocks. Time to combine them on **real websites**.
Each project adds one more layer of complexity.

---

## 🗣️ Mini Project 1 — Quotes Scraper (`quotes.toscrape.com`)

**Goal**: Scrape all quotes, authors, and tags from one page and store as a list of dicts.

### Step 1 — Inspect the Page First!

Go to `https://quotes.toscrape.com` → Right-click a quote → Inspect:

```
What you see on screen        What DevTools shows
─────────────────────────────────────────────────────
"The world as we..."     ->  <span class="text">"The world as we..."
by Albert Einstein       ->  <small class="author">Albert Einstein</small>
Tags: change, deep...    ->  <a class="tag">change</a>
                              <a class="tag">deep-thoughts</a>
```

Each quote lives inside a `<div class="quote">` container.

### Scraping Strategy
```
1. Find all  <div class="quote">          <- one container per quote
2. Inside each, find:
     <span class="text">                  <- the quote text
     <small class="author">              <- the author name
     <a class="tag">  (all of them)      <- all tags
3. Store each as a dict -> list of dicts
```

In [1]:
# MINI PROJECT 1 — Quotes Scraper (Single Page)

import requests
import time
from bs4 import BeautifulSoup

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'
}

# Step 1: Fetch the page
url = 'https://quotes.toscrape.com'
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f'Failed: {e}')
    response = None

if response:
    soup = BeautifulSoup(response.text, 'html.parser')

    # Step 2: Find all quote containers
    quote_boxes = soup.find_all('div', class_='quote')
    print(f'Found {len(quote_boxes)} quotes on this page\n')

    # Step 3: Extract data from each container
    all_quotes = []

    for box in quote_boxes:

        # None-safe extraction (applying what we learned!)
        text_tag   = box.find('span', class_='text')
        author_tag = box.find('small', class_='author')
        tag_links  = box.find_all('a', class_='tag')

        quote_text = text_tag.get_text(strip=True)   if text_tag   else 'N/A'
        author     = author_tag.get_text(strip=True) if author_tag else 'N/A'
        tags       = [a.get_text(strip=True) for a in tag_links]   # list of tags

        all_quotes.append({
            'quote' : quote_text,
            'author': author,
            'tags'  : tags
        })

    # Step 4: Print the results
    for i, q in enumerate(all_quotes, 1):
        print(f'Quote {i}:')
        print(f'  Text   : {q["quote"][:60]}...')
        print(f'  Author : {q["author"]}')
        print(f'  Tags   : {q["tags"]}')
        print()

Found 10 quotes on this page

Quote 1:
  Text   : “The world as we have created it is a process of our thinkin...
  Author : Albert Einstein
  Tags   : ['change', 'deep-thoughts', 'thinking', 'world']

Quote 2:
  Text   : “It is our choices, Harry, that show what we truly are, far ...
  Author : J.K. Rowling
  Tags   : ['abilities', 'choices']

Quote 3:
  Text   : “There are only two ways to live your life. One is as though...
  Author : Albert Einstein
  Tags   : ['inspirational', 'life', 'live', 'miracle', 'miracles']

Quote 4:
  Text   : “The person, be it gentleman or lady, who has not pleasure i...
  Author : Jane Austen
  Tags   : ['aliteracy', 'books', 'classic', 'humor']

Quote 5:
  Text   : “Imperfection is beauty, madness is genius and it's better t...
  Author : Marilyn Monroe
  Tags   : ['be-yourself', 'inspirational']

Quote 6:
  Text   : “Try not to become a man of success. Rather become a man of ...
  Author : Albert Einstein
  Tags   : ['adulthood', 'success', 'value']


## 📚 Mini Project 2 — Books Scraper (`books.toscrape.com`)

**Goal**: Scrape title, price, rating, and availability — then convert price to a number for sorting.

### Inspect the Page First!

Go to `https://books.toscrape.com` → Inspect a book card:

```
What DevTools shows:
────────────────────────────────────────────────────────────
<article class="product_pod">
  <p class="star-rating Three">                <- rating in class name!
  <h3><a title="A Light in the Attic">         <- title in 'title' attribute
  <p class="price_color">£51.77</p>            <- price
  <p class="availability">In stock</p>         <- availability
</article>
```

### Key Trick: Rating is Hidden in the Class Name
```python
# <p class="star-rating Three"> means 3 stars
# The second class word IS the rating!
rating_tag = book.find('p', class_='star-rating')
rating = rating_tag['class'][1]    # ['star-rating', 'Three'] -> 'Three'
```

In [2]:
# MINI PROJECT 2 — Books Scraper

url = 'https://books.toscrape.com'
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f'Failed: {e}')
    response = None

if response:
    soup = BeautifulSoup(response.text, 'html.parser')

    book_cards = soup.find_all('article', class_='product_pod')
    print(f'Found {len(book_cards)} books\n')

    all_books = []

    for card in book_cards:

        # Title is in the 'title' attribute of <a> inside <h3>
        h3 = card.find('h3')
        title = h3.find('a')['title'] if h3 and h3.find('a') else 'N/A'

        # Price — comes as '£51.77', convert to float
        price_tag = card.find('p', class_='price_color')
        if price_tag:
            price_str = price_tag.get_text(strip=True)          # '£51.77'
            price_num = float(price_str.replace('£', '').replace('Â', '').strip())
        else:
            price_num = 0.0

        # Rating — hidden in class name: 'star-rating Three'
        rating_tag = card.find('p', class_='star-rating')
        rating = rating_tag['class'][1] if rating_tag else 'N/A'  # 'One','Two'...'Five'

        # Availability
        avail_tag = card.find('p', class_='availability')
        availability = avail_tag.get_text(strip=True) if avail_tag else 'N/A'

        all_books.append({
            'title'       : title,
            'price'       : price_num,
            'rating'      : rating,
            'availability': availability
        })

    # Print first 5 books
    print('First 5 books scraped:')
    print('-' * 55)
    for book in all_books[:5]:
        print(f"  {book['title'][:35]:<35} | {book['price']:>6.2f} | {book['rating']}")

    print()

    # Bonus: sort by price and find cheapest
    sorted_books = sorted(all_books, key=lambda x: x['price'])
    print(f"Cheapest book : {sorted_books[0]['title']} @ £{sorted_books[0]['price']}")
    print(f"Most expensive: {sorted_books[-1]['title']} @ £{sorted_books[-1]['price']}")

Found 20 books

First 5 books scraped:
-------------------------------------------------------
  A Light in the Attic                |  51.77 | Three
  Tipping the Velvet                  |  53.74 | One
  Soumission                          |  50.10 | One
  Sharp Objects                       |  47.82 | Four
  Sapiens: A Brief History of Humanki |  54.23 | Five

Cheapest book : Starving Hearts (Triangular Trade Trilogy, #1) @ £13.99
Most expensive: Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991 @ £57.25


## 🔄 Mini Project 3 — Pagination Scraper (Multiple Pages)

**Goal**: Scrape ALL quotes across ALL pages of `quotes.toscrape.com` — automatically.

### How Pagination Works

Most websites have a **Next** button to go to the next page:

```html
<li class="next">
    <a href="/page/2/">Next</a>     <- href tells us the next page URL
</li>
```

### Pagination Strategy

```
1. Start at page 1
2. Scrape all quotes on the page
3. Find the 'Next' button
4. If Next exists -> go to next page URL -> repeat from step 2
5. If Next does NOT exist -> we are on the last page -> STOP
```

```python
# The loop pattern:
base_url = 'https://quotes.toscrape.com'
next_url = '/'

while next_url:                          # Keep going while there's a next page
    soup = fetch(base_url + next_url)
    # ... scrape this page ...

    next_btn = soup.find('li', class_='next')
    next_url = next_btn.find('a')['href'] if next_btn else None   # None = stop
```

> **Always add `time.sleep()` inside pagination loops** — you're hitting the same server many times in a row!

In [ ]:
# MINI PROJECT 3 — Full Pagination Scraper + Save to JSON

import json
import time
import random

base_url = 'https://quotes.toscrape.com'
next_url = '/'
all_quotes = []
page_num = 1

print('Starting paginated scrape...')
print('-' * 40)

while next_url:

    full_url = base_url + next_url

    # Safe fetch
    try:
        response = requests.get(full_url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error on page {page_num}: {e}')
        break

    soup = BeautifulSoup(response.text, 'html.parser')

    # Scrape all quotes on this page
    boxes = soup.find_all('div', class_='quote')

    for box in boxes:
        text_tag   = box.find('span', class_='text')
        author_tag = box.find('small', class_='author')
        tag_links  = box.find_all('a', class_='tag')

        all_quotes.append({
            'quote' : text_tag.get_text(strip=True)   if text_tag   else 'N/A',
            'author': author_tag.get_text(strip=True) if author_tag else 'N/A',
            'tags'  : [a.get_text(strip=True) for a in tag_links],
            'page'  : page_num
        })

    print(f'Page {page_num:2d} | Scraped {len(boxes)} quotes | Total so far: {len(all_quotes)}')

    # Find Next button
    next_li  = soup.find('li', class_='next')
    next_url = next_li.find('a')['href'] if next_li else None

    page_num += 1

    # Be polite - random delay between pages
    if next_url:
        time.sleep(random.uniform(1, 2))

print('-' * 40)
print(f'Done! Total quotes scraped: {len(all_quotes)}')
print()

# Save to JSON
output_file = 'all_quotes.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_quotes, f, indent=4, ensure_ascii=False)

print(f'Saved to {output_file}')

# Quick stats
from collections import Counter
top_authors = Counter(q['author'] for q in all_quotes).most_common(3)
print('\nTop 3 authors:')
for author, count in top_authors:
    print(f'  {author}: {count} quotes')

## 🖼️ Bonus — Downloading Images from a Website

Scraping isn't just text — you can also download images. The trick:

- Image URLs are in `<img>` tags, in the `src` attribute
- Use `response.content` (not `.text`) for binary files like images

```python
# Find image src
img_url = soup.find('img')['src']

# Download image as bytes
img_response = requests.get(img_url)

# Save to disk
with open('image.jpg', 'wb') as f:       # 'wb' = write binary
    f.write(img_response.content)         # .content not .text!
```

### Handling Relative vs Absolute URLs

```
Absolute URL:  https://example.com/images/photo.jpg  <- ready to use
Relative URL:  /images/photo.jpg                     <- need to add base_url

base_url = 'https://example.com'
full_url = base_url + relative_src      # -> https://example.com/images/photo.jpg
```

In [ ]:
# BONUS — Download Images from a Website

import os

# Create a folder to save images
os.makedirs('downloaded_images', exist_ok=True)

# Scrape books.toscrape.com for book cover images
url = 'https://books.toscrape.com'
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f'Failed: {e}')
    response = None

if response:
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all images (limit to first 3 to be quick)
    images = soup.find_all('img')[:3]

    for i, img in enumerate(images, 1):
        src = img.get('src', '')                    # e.g. 'media/cache/xx/yy/photo.jpg'

        # books.toscrape uses relative URLs -> add base
        if src.startswith('http'):
            img_url = src                           # already absolute
        else:
            img_url = 'https://books.toscrape.com/' + src.lstrip('../')

        # Download image
        try:
            img_resp = requests.get(img_url, headers=headers, timeout=10)
            img_resp.raise_for_status()

            filename = f'downloaded_images/book_{i}.jpg'
            with open(filename, 'wb') as f:
                f.write(img_resp.content)           # .content for binary!

            print(f'Downloaded: {filename} ({len(img_resp.content)} bytes)')

        except requests.exceptions.RequestException as e:
            print(f'Failed to download image {i}: {e}')

        time.sleep(0.5)    # Small delay between image downloads

    print('\nDone! Check the downloaded_images/ folder.')

## 🐛 Debugging — My Scraper Isn't Working!

Every beginner hits a wall where the code runs but gives **empty results or crashes**.
Don't panic. Follow this checklist step by step:

```
Step 1  Check status_code
        200 -> OK   |   403 -> Blocked   |   404 -> Wrong URL

Step 2  Print response.text[:1000]
        Is your data visible in the raw HTML?
        If NO data is visible -> the page uses JavaScript (need Selenium)

Step 3  Add User-Agent header
        Got 403 or empty page? Server is blocking Python requests.
        Add headers = {'User-Agent': 'Mozilla/5.0 ...'}

Step 4  Print soup.prettify()
        Visualize the full HTML structure to find the right tags.

Step 5  Check your selector
        Print soup.find('div', class_='xyz') and see what it returns.
        None means wrong tag or class name — re-inspect the website.

Step 6  Check for None before .text
        tag = soup.find(...)
        print(tag)        <- prints None if not found
        print(tag.text)   <- crashes if tag is None!
```

### Most Common Reasons Scraper Returns Empty / Wrong Data

| Symptom | Likely Cause | Fix |
|---------|--------------|-----|
| `find()` returns `None` | Wrong class/tag name | Re-inspect element |
| `response.text` has no data | JS-rendered page | Use Selenium (NB2) |
| Status code 403 | Server blocking you | Add User-Agent header |
| Weird characters in text | Encoding issue | Set `response.encoding = 'utf-8'` |
| Data missing on some pages | Tag absent on that page | Use None safety pattern |

In [ ]:
# DEBUGGING CELL — Run this checklist when scraper isn't working

url = 'https://quotes.toscrape.com'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'
}

response = requests.get(url, headers=headers, timeout=10)

# --- Step 1: Check status code ---
print(f'Step 1 | Status Code : {response.status_code}')
print(f'        200=OK | 403=Blocked | 404=Not Found | 500=Server Error')
print()

# --- Step 2: Is the data in raw HTML? ---
print('Step 2 | First 500 chars of HTML:')
print('-' * 50)
print(response.text[:500])
print('-' * 50)
print('       Tip: Search for a known keyword (e.g. a quote text)')
print(f'       "The world" in HTML: {"The world" in response.text}')
print()

# --- Step 3: Check encoding ---
print(f'Step 3 | Encoding : {response.encoding}')
# If you see garbled text, force UTF-8:
# response.encoding = 'utf-8'
print()

soup = BeautifulSoup(response.text, 'html.parser')

# --- Step 4: Test your selector directly ---
print('Step 4 | Testing selector soup.find("div", class_="quote"):')
test = soup.find('div', class_='quote')
print(f'        Result : {str(test)[:80]}...' if test else '        Result : None <- WRONG selector!')
print()

# --- Step 5: Prettify a small section to understand structure ---
print('Step 5 | Structure of first quote div:')
if test:
    print(test.prettify()[:400])

## 🧰 3 More Essential Tools

### 1. `response.json()` — When a Website Returns JSON, Not HTML

Many modern websites expose **public APIs** that return JSON data directly.
For these, you don't need BS4 at all — `requests` alone is enough:

```python
r = requests.get('https://api.example.com/books')
data = r.json()   # Directly returns a Python dict or list — no BS4 needed!
print(data['books'][0]['title'])
```

> **How to tell if a URL returns JSON?**  
> Open it in browser → if you see `{ }` or `[ ]` with text, it's a JSON API.

---

### 2. `urljoin` — The Clean Way to Handle Relative URLs

Scraped `href` and `src` values are often **relative** (no domain).
Instead of manually joining strings, use `urljoin` — it handles all edge cases:

```python
from urllib.parse import urljoin

base = 'https://books.toscrape.com'
relative = '../media/cache/photo.jpg'

# WRONG way (fragile):
full = base + '/' + relative           # breaks with '../' paths

# RIGHT way:
full = urljoin(base, relative)         # always correct!
```

---

### 3. `requests.Session()` — Set Headers Once, Use Everywhere

When scraping multiple pages, passing `headers=headers` in every `requests.get()` is repetitive.
A `Session` persists headers (and cookies) across all requests automatically:

```python
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 ...'})

r1 = session.get('https://site.com/page/1/')  # headers applied automatically
r2 = session.get('https://site.com/page/2/')  # same headers, no extra work

session.close()  # always close when done
```

> **Bonus**: Session also persists **cookies** — useful for scraping sites after login.

In [ ]:
# 3 ESSENTIAL TOOLS IN ACTION

from urllib.parse import urljoin

# ─────────────────────────────────────────────
# Tool 1: response.json() — scraping a JSON API
# ─────────────────────────────────────────────
print('=' * 50)
print('Tool 1: response.json()')
print('=' * 50)

# quotes.toscrape.com also has a JSON-like page we can use
# Using a real free public API for demo
api_url = 'https://httpbin.org/json'
r = requests.get(api_url, timeout=10)

print(f'Status     : {r.status_code}')
print(f'Content-Type: {r.headers["Content-Type"]}')
print()

data = r.json()             # No BS4 needed! Direct dict.
print('Parsed JSON:')
print(data)
print()

# ─────────────────────────────────────────────
# Tool 2: urljoin — clean URL building
# ─────────────────────────────────────────────
print('=' * 50)
print('Tool 2: urljoin')
print('=' * 50)

base = 'https://books.toscrape.com/catalogue/'

# Typical relative paths scraped from websites
relatives = [
    'page-2.html',
    '../media/cache/2/ae/photo.jpg',
    '/about/',
    'https://other.com/page'       # already absolute -> urljoin keeps it
]

for rel in relatives:
    full = urljoin(base, rel)
    print(f'  {rel:<40} -> {full}')
print()

# ─────────────────────────────────────────────
# Tool 3: requests.Session()
# ─────────────────────────────────────────────
print('=' * 50)
print('Tool 3: requests.Session()')
print('=' * 50)

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'
})

# All requests now use the same headers automatically
pages = [1, 2, 3]
for page in pages:
    r = session.get(f'https://quotes.toscrape.com/page/{page}/', timeout=10)
    soup = BeautifulSoup(r.text, 'html.parser')
    quote_count = len(soup.find_all('div', class_='quote'))
    print(f'  Page {page} | Status: {r.status_code} | Quotes: {quote_count}')
    import time
    time.sleep(1)

session.close()   # always close!
print('Session closed.')

---
# ✅ Notebook 1 — Complete!

## What You've Mastered

```
REQUESTS
  [x] requests.get()          Fetch any webpage
  [x] status_code             Check if request succeeded
  [x] headers / User-Agent    Avoid getting blocked
  [x] params                  Navigate pages cleanly
  [x] try/except              Handle errors gracefully
  [x] time.sleep()            Be polite to servers

BEAUTIFULSOUP4
  [x] BeautifulSoup()         Parse raw HTML into a tree
  [x] find()                  Get the first matching tag
  [x] find_all()              Get all matching tags -> list
  [x] select() / select_one() CSS selector search
  [x] tag.text / get_text()   Extract text content
  [x] tag.get('attr')         Extract attributes safely
  [x] .parent / .children     Navigate the HTML tree
  [x] find_next()             Find next sibling tag
  [x] Regex in find_all()     Pattern-based searching
  [x] Text cleaning           strip(), replace(), split()
  [x] String -> Number        Convert scraped text to float
  [x] None safety             Never crash on missing tags
  [x] Scraping tables         th, tr, td -> list of dicts

SAVING DATA
  [x] CSV                     For tabular data
  [x] JSON                    For nested/flexible data
  [x] Images                  Download binary files

REAL PROJECTS
  [x] Quotes scraper          Single page + structured data
  [x] Books scraper           Price conversion + sorting
  [x] Pagination scraper      All pages + save to JSON
  [x] Image downloader        Binary file download
```

---

## 🎯 What's in Notebook 2

| Topic | What You'll Learn |
|-------|------------------|
| Selenium Setup | Control a real browser with Python |
| JS-rendered pages | Scrape sites that requests cannot handle |
| Clicking & Forms | Automate button clicks and input filling |
| Waits | Handle slow-loading pages correctly |
| Infinite Scroll | Scrape pages that load on scroll |
| Selenium + BS4 | Combine both for best results |
| Full Projects | 2 end-to-end projects using Selenium |

> Open `02_Selenium.ipynb` when you are ready!